In [137]:
import pandas as pd
import re
import os
import numpy as np
import requests
import matplotlib.pyplot as plt

In [113]:
os.getcwd()
#Num format
pd.options.display.float_format = '{:,.2f}'.format

path="scraping_output/final"

In [114]:
def clean_data(df):
    # Lowercase name and size
    df['name'] = df['name'].str.lower()
    df['size'] = df['size'].str.lower()

    # Infer type from name when type is NaN
    apt_kw  = r'departamento|depto|dpto|apto|apartamento|piso|flat|loft|studio|estudio'
    house_kw = r'\bcasa\b|chalet|villa|townhouse|bungalow'
    mask_nan = df['type'].isna()
    df.loc[mask_nan & df['name'].str.contains(apt_kw,   case=False, na=False), 'type'] = 'Apartment'
    df.loc[mask_nan & df['name'].str.contains(house_kw, case=False, na=False), 'type'] = 'House'

    #Keep apartments and houses
    df = df[df['type'].isin(['Apartment', 'House'])]

    def parse_number(val):
        """Parse number string handling dot/comma as thousand or decimal separator."""
        match = re.match(r'^[\d.,]+', str(val).strip())
        if not match:
            return np.nan
        s = match.group()

        has_dot = '.' in s
        has_comma = ',' in s

        if has_dot and has_comma:
            if s.rfind('.') > s.rfind(','):
                # e.g. 1,047.89 → comma=miles, dot=decimal
                s = s.replace(',', '')
            else:
                # e.g. 1.047,89 → dot=miles, comma=decimal
                s = s.replace('.', '').replace(',', '.')
        elif has_dot:
            # dot=thousands if ALL groups after dots have exactly 3 digits
            parts = s.split('.')
            if all(len(p) == 3 for p in parts[1:]):
                s = s.replace('.', '')
        elif has_comma:
            parts = s.split(',')
            if all(len(p) == 3 for p in parts[1:]):
                s = s.replace(',', '')
            else:
                s = s.replace(',', '.')

        return pd.to_numeric(s, errors='coerce')

    # Extract currency prefix (e.g. "USD") from price string and update currency column
    def extract_currency(row):
        val = row['price']
        if pd.isna(val):
            return row['currency'], val
        s = str(val).strip()
        m = re.match(r'^([A-Z]{2,4})\s*([\d.,].*)$', s)
        if m:
            detected = m.group(1)
            price_str = m.group(2)
            # Keep existing currency if already set, otherwise use detected one
            currency = row['currency'] if pd.notna(row['currency']) else detected
            return currency, price_str
        return row['currency'], val

    extracted = df[['price', 'currency']].apply(extract_currency, axis=1, result_type='expand')
    df['currency'] = extracted[0]
    df['price'] = extracted[1]

    # Convert price to numeric: remove "$" then parse
    def parse_price(val):
        if pd.isna(val):
            return np.nan
        return parse_number(str(val).replace('$', ''))

    # Convert size to numeric: remove "m²"/"m2" then parse
    def parse_size(val):
        if pd.isna(val):
            return np.nan
        return parse_number(str(val).replace('m²', '').replace('m2', ''))

    df['price'] = df['price'].apply(parse_price)
    df['size'] = df['size'].apply(parse_size)

    # Si el país es Uruguay, Paraguay o Bolivia, renombra price_usd a price y m2 a size
    special_countries = {"Uruguay", "Paraguay", "Bolivia"}
    df["price"] = df.apply(
        lambda row: row["price_usd"] if row["country"] in special_countries and pd.notna(row["price_usd"]) else row["price"],
        axis=1,
    )
    df["size"] = df.apply(
        lambda row: row["m2"] if row["country"] in special_countries and pd.notna(row["m2"]) else row["size"],
        axis=1,
    )
    # Añadir source "Infocasas" si país es Uruguay, Paraguay o Bolivia
    df["source"] = df.apply(
        lambda row: "Infocasas" if row["country"] in special_countries else row["source"],
        axis=1,
    )
    # Drop rows where price or size is NaN
    df = df.dropna(subset=['price', 'size'])
    #Dejar primera coincidencia de nombre y precio
    df = df.drop_duplicates(subset=['name', 'price'], keep='first')

    return df


In [115]:
##Load raw data

raw=pd.read_csv("scraping_output/raw/total_scraping.csv", low_memory=False)
print("Total rows", len(raw))
raw.head()

Total rows 94125


,name,price,currency,type,size,bedrooms,bathrooms,latitude,longitude,street,region,locality,consultation_date,source,country,operation,price_usd,rooms,m2
0,Departamento en Venta en La Carolina,$ 140,NaN,Apartment,70 m²,1.0,2.0,-0.19,-78.49,"Eloy Alfaro Y de la Republica, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN
1,Departamento en Venta en San Isidro del Inca,$ 69.900,NaN,Apartment,69 m²,3.0,2.0,-0.14,-78.46,"Calle N50G, Quito, ECU",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN
2,Departamento en Venta en Centro De Quito,$ 68.000,NaN,Apartment,106 m²,3.0,2.0,-0.21,-78.50,"Asunción & Venezuela, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN
3,Oficina en Venta en La Mariscal,$ 140.000,NaN,Accommodation,166 m²,7.0,2.0,-0.20,-78.49,Av. Francisco de Orellana & Avenida 10 de Agos...,Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN
4,Departamento en Venta en Nayón,$ 110.000,NaN,Apartment,115 m²,2.0,3.0,-0.16,-78.44,"Jaime Roldos Aguilera & 19 de Diciembre, Quito...",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN


In [116]:
raw["currency"].unique()

array([nan, 'BRL'], dtype=object)

In [117]:
print("Data before cleaning")
print(raw.groupby(['country', 'operation']).size().unstack(fill_value=0).to_string())

Data before cleaning
operation    rent  sell
country                
Argentina    3000  3000
Bolivia      2100  2100
Brazil       1200  1200
Chile        3000  3000
Colombia     3000  3000
Costa-rica   3849  4000
Dominicana   1667  3689
Ecuador      3000  3000
El-salvador   666  1321
Guatemala    2764  4000
Honduras     1213  1652
Mexico       5162  7766
Nicaragua    1807  2142
Panama       3427  4000
Paraguay     2100  2100
Peru         3000  3000
Uruguay      2100  2100


In [118]:
## Clean data
clean_scrap=clean_data(raw.copy())
print("Total rows:", len(clean_scrap))
clean_scrap

Total rows: 47789


,name,price,currency,type,size,bedrooms,bathrooms,latitude,longitude,street,region,locality,consultation_date,source,country,operation,price_usd,rooms,m2
0,departamento en venta en la carolina,140.00,NaN,Apartment,70.00,1.0,2.0,-0.19,-78.49,"Eloy Alfaro Y de la Republica, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN
1,departamento en venta en san isidro del inca,"69,900.00",NaN,Apartment,69.00,3.0,2.0,-0.14,-78.46,"Calle N50G, Quito, ECU",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN
2,departamento en venta en centro de quito,"68,000.00",NaN,Apartment,106.00,3.0,2.0,-0.21,-78.50,"Asunción & Venezuela, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN
4,departamento en venta en nayón,"110,000.00",NaN,Apartment,115.00,2.0,3.0,-0.16,-78.44,"Jaime Roldos Aguilera & 19 de Diciembre, Quito...",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN
5,casa en venta en garzota,"350,000.00",NaN,House,622.00,15.0,9.0,-2.15,-79.89,"Ciudadela Ietel, IETEL, Guayaquil, Ecuador",Guayas,Guayaquil,2026-03-19 18:26:04,Properati,Ecuador,sell,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94119,departamento en alquiler de 3 dormitorios - eq...,"1,428.00",NaN,Apartment,1.00,NaN,1.0,NaN,NaN,NaN,Equipetrol,Apartamento en,2026-04-28,Infocasas,Bolivia,rent,"1,428.00",3.00,1.00
94120,casa en alquiler zona norte,431.00,NaN,Apartment,100.00,NaN,NaN,NaN,NaN,NaN,Norte,Casa en,2026-04-28,Infocasas,Bolivia,rent,431.00,NaN,100.00
94121,"departamento de 2hab. en alquiler, zona norte",650.00,NaN,Apartment,82.00,NaN,NaN,NaN,NaN,NaN,Norte,Apartamento en,2026-04-28,Infocasas,Bolivia,rent,650.00,2.00,82.00
94122,departamento en alquiler de 2 dormitorios – av...,750.00,NaN,Apartment,89.00,NaN,NaN,NaN,NaN,NaN,Norte,Apartamento en,2026-04-28,Infocasas,Bolivia,rent,750.00,2.00,89.00


In [119]:
# currency_map = {
#     'Argentina': 'ARS', 'Brazil': 'BRL', 'Chile': 'CLP',
#     'Colombia': 'COP', 'Costa-rica': 'CRC', 'Ecuador': 'USD',
#     'El-salvador': 'USD', 'Guatemala': 'GTQ', 'Honduras': 'HNL',
#     'Mexico': 'MXN', 'Nicaragua': 'NIO', 'Panama': 'USD', 'Peru': 'PEN'
# }

In [120]:
# # Get exchange rates (exchangerate-api)
# API_KEY="cf53b6293b066860dd66b072"
# url = f"https://v6.exchangerate-api.com/v6/{API_KEY}/latest/USD"
# r = requests.get(url).json()

# if r.get('result') == 'success':
#     rates = r['conversion_rates']
#     last_update = pd.to_datetime(r['time_last_update_utc']).strftime('%Y-%m-%d')

#     exchange_table = pd.DataFrame([
#         {
#             'country': country,
#             'rate': 1.0 if code == 'USD' else (1 / rates[code] if code in rates else None),
#             'last_update': last_update
#         }
#         for country, code in currency_map.items()
#     ])

#     print(exchange_table.to_string(index=False))
# else:
#     print("⚠️ Error al obtener tasas:", r)


# def get_latest_exchange_rate(country_code):
#     """
#     Devuelve el último tipo de cambio disponible (moneda local por 1 USD)
#     usando World Bank API (indicador PA.NUS.FCRF).

#     Parámetros
#     ----------
#     country_code : str  — ISO2 (e.g. 'MX', 'BR', 'CL')

#     Retorna
#     -------
#     dict con keys: country_code, source, date, exchange_rate
#     """
#     country_code = country_code.upper()
#     url = (
#         f"https://api.worldbank.org/v2/country/{country_code}"
#         f"/indicator/PA.NUS.FCRF"
#         f"?format=json&per_page=10&mrv=1"
#     )
#     r = requests.get(url)
#     r.raise_for_status()
#     records = r.json()[1]
#     if not records:
#         raise ValueError(f"Sin datos World Bank para {country_code}")
#     item = records[0]
#     return {
#         "country_code": country_code,
#         "source": "worldbank",
#         "date": item["date"],
#         "exchange_rate": item["value"],
#     }


# # ISO2 codes per country (USD countries get rate=1.0 directly)
# iso2_map = {
#     'Argentina':    'AR',
#     'Brazil':       'BR',
#     'Chile':        'CL',
#     'Colombia':     'CO',
#     'Costa-rica':   'CR',
#     'Ecuador':      'EC',
#     'El-salvador':  'SV',
#     'Guatemala':    'GT',
#     'Honduras':     'HN',
#     'Mexico':       'MX',
#     'Nicaragua':    'NI',
#     'Panama':       'PA',
#     'Peru':         'PE',
# }

# rows = []
# for country, iso2 in iso2_map.items():
#     if iso2 == 'USD':
#         rows.append({
#             'country':              country,
#             'lcu_per_usd':          1.0,    # local currency units per 1 USD (World Bank raw)
#             'rate':                 1.0,    # USD per 1 local unit (used for price conversion)
#             'last_update':          None,
#         })
#     else:
#         wb = get_latest_exchange_rate(iso2)
#         rows.append({
#             'country':              country,
#             'lcu_per_usd':          wb['exchange_rate'],          # World Bank raw value
#             'rate':                 1 / wb['exchange_rate'],      # USD per local unit
#             'last_update':          wb['date'],
#         })

# exchange_table = pd.DataFrame(rows)
# exchange_table

In [121]:
# Get exchange rates from IMF CSV — indicator: US Dollar per domestic currency, Period average
# Fallback per country: Monthly → Quarterly → Annual

COUNTRY_TO_IMF = {
    'Argentina':   'Argentina',
    'Brazil':      'Brazil',
    'Chile':       'Chile',
    'Colombia':    'Colombia',
    'Costa-rica':  'Costa Rica',
    'Dominicana':  'Dominican Republic',
    'Guatemala':   'Guatemala',
    'Honduras':    'Honduras',
    'Mexico':      'Mexico',
    'Nicaragua':   'Nicaragua',
    'Peru':        'Peru',
    'Ecuador':     'Ecuador',
    'El-salvador': 'El Salvador',
    'Panama':      'Panama',
    'Uruguay':    'Uruguay',
    "Paraguay":   "Paraguay",
    "Bolivia":   "Bolivia"
}

imf_csv = pd.read_csv(
    "../exchange_rates_imf.csv",
    on_bad_lines="skip",
    engine="python",
    encoding="utf-8",
    usecols=["COUNTRY", "INDICATOR", "TYPE_OF_TRANSFORMATION", "FREQUENCY", "TIME_PERIOD", "OBS_VALUE"],
)

def parse_imf_period(val):
    if "-M" in str(val):
        return pd.to_datetime(val, format="%Y-M%m")
    if "-Q" in str(val):
        year, q = val.split("-Q")
        return pd.Timestamp(year=int(year), month=(int(q) - 1) * 3 + 1, day=1)
    try:
        return pd.Timestamp(year=int(val), month=1, day=1)
    except Exception:
        return pd.NaT

imf_base = (
    imf_csv
    .loc[
        (imf_csv["INDICATOR"] == "US Dollar per domestic currency") &
        (imf_csv["TYPE_OF_TRANSFORMATION"] == "Period average") &
        (imf_csv["COUNTRY"].isin(COUNTRY_TO_IMF.values()))
    ]
    .assign(
        period=lambda d: d["TIME_PERIOD"].map(parse_imf_period),
        rate=lambda d: pd.to_numeric(d["OBS_VALUE"], errors="coerce"),
    )
    .dropna(subset=["rate", "period"])
)

def last_by_freq(freq):
    sub = imf_base[imf_base["FREQUENCY"] == freq]
    if sub.empty:
        return {}
    return (
        sub.loc[sub.groupby("COUNTRY")["period"].idxmax()]
           .set_index("COUNTRY")[["rate", "period"]]
           .to_dict("index")
    )

freq_data = {f: last_by_freq(f) for f in ["Monthly", "Quarterly", "Annual"]}

rows = []
for country, imf_name in COUNTRY_TO_IMF.items():
    for freq in ["Monthly", "Quarterly", "Annual"]:
        if imf_name in freq_data[freq]:
            rec = freq_data[freq][imf_name]
            rows.append({
                "country":     country,
                "rate":        rec["rate"],
                "last_update": rec["period"].strftime("%Y-%m"),
            })
            break

exchange_table = pd.DataFrame(rows).sort_values("country").reset_index(drop=True)
exchange_table.style.format({"rate": "{:.6f}"})

,country,rate,last_update
0,Argentina,0.000806,2025-01
1,Bolivia,0.144718,2026-01
2,Brazil,0.178978,2025-01
3,Chile,0.001051,2025-01
4,Colombia,0.000264,2025-12
5,Costa-rica,0.002131,2026-03
6,Dominicana,0.015746,2026-01
7,Ecuador,1.000000,2026-03
8,El-salvador,1.000000,2026-03
9,Guatemala,0.130646,2026-03


In [122]:
clean_scrap = clean_scrap.merge(
    exchange_table[['country', 'rate', 'last_update']],
    on='country',
    how='left'
)

# Create price_usd:
# Create price_usd:
# - currency == 'USD' → direct price
# - currency == 'UF'  → price * uf_value (CLP) * rate_to_usd del país (CLP→USD)
# - NaN o moneda local → price * rate_to_usd
clean_scrap['price_usd'] = clean_scrap.apply(
    lambda row: row['price'] if row['currency'] == 'USD' or row['country'] in {'Uruguay', 'Paraguay', 'Bolivia'}
    #UF from Banco Central de Chile
    #https://tinyurl.com/yz5ffy3n
                else row['price'] *39841.72* row['rate'] if row['currency'] == 'UF'
                else row['price'] * row['rate'],
    axis=1
)

#Price per square meter
clean_scrap['price_per_sq_meter'] = clean_scrap['price_usd'] / clean_scrap['size']


clean_scrap

,name,price,currency,type,size,bedrooms,bathrooms,latitude,longitude,street,...,consultation_date,source,country,operation,price_usd,rooms,m2,rate,last_update,price_per_sq_meter
0,departamento en venta en la carolina,140.00,NaN,Apartment,70.00,1.0,2.0,-0.19,-78.49,"Eloy Alfaro Y de la Republica, Quito, Ecuador",...,2026-03-19 18:26:04,Properati,Ecuador,sell,140.00,NaN,NaN,1.00,2026-03,2.00
1,departamento en venta en san isidro del inca,"69,900.00",NaN,Apartment,69.00,3.0,2.0,-0.14,-78.46,"Calle N50G, Quito, ECU",...,2026-03-19 18:26:04,Properati,Ecuador,sell,"69,900.00",NaN,NaN,1.00,2026-03,"1,013.04"
2,departamento en venta en centro de quito,"68,000.00",NaN,Apartment,106.00,3.0,2.0,-0.21,-78.50,"Asunción & Venezuela, Quito, Ecuador",...,2026-03-19 18:26:04,Properati,Ecuador,sell,"68,000.00",NaN,NaN,1.00,2026-03,641.51
3,departamento en venta en nayón,"110,000.00",NaN,Apartment,115.00,2.0,3.0,-0.16,-78.44,"Jaime Roldos Aguilera & 19 de Diciembre, Quito...",...,2026-03-19 18:26:04,Properati,Ecuador,sell,"110,000.00",NaN,NaN,1.00,2026-03,956.52
4,casa en venta en garzota,"350,000.00",NaN,House,622.00,15.0,9.0,-2.15,-79.89,"Ciudadela Ietel, IETEL, Guayaquil, Ecuador",...,2026-03-19 18:26:04,Properati,Ecuador,sell,"350,000.00",NaN,NaN,1.00,2026-03,562.70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47784,departamento en alquiler de 3 dormitorios - eq...,"1,428.00",NaN,Apartment,1.00,NaN,1.0,NaN,NaN,NaN,...,2026-04-28,Infocasas,Bolivia,rent,"1,428.00",3.00,1.00,0.14,2026-01,"1,428.00"
47785,casa en alquiler zona norte,431.00,NaN,Apartment,100.00,NaN,NaN,NaN,NaN,NaN,...,2026-04-28,Infocasas,Bolivia,rent,431.00,NaN,100.00,0.14,2026-01,4.31
47786,"departamento de 2hab. en alquiler, zona norte",650.00,NaN,Apartment,82.00,NaN,NaN,NaN,NaN,NaN,...,2026-04-28,Infocasas,Bolivia,rent,650.00,2.00,82.00,0.14,2026-01,7.93
47787,departamento en alquiler de 2 dormitorios – av...,750.00,NaN,Apartment,89.00,NaN,NaN,NaN,NaN,NaN,...,2026-04-28,Infocasas,Bolivia,rent,750.00,2.00,89.00,0.14,2026-01,8.43


In [123]:
clean_scrap["currency"].unique()

array([nan, 'USD', 'BRL', 'UF'], dtype=object)

In [147]:
clean_scrap["type"].unique()

array(['Apartment', 'House'], dtype=object)

In [124]:
print("\nData after cleaning")
print(clean_scrap.groupby(['country', 'type', 'operation']).size().unstack(fill_value=0).to_string())


Data after cleaning
operation              rent  sell
country     type                 
Argentina   Apartment   421   451
            House       265   461
Bolivia     Apartment   584   548
            House       911   875
Brazil      Apartment    10    19
            House         3     4
Chile       Apartment  1537  2731
Colombia    Apartment   707   471
            House       328   748
Costa-rica  Apartment   856  1436
            House       766  1269
Dominicana  Apartment   322  1274
            House        23   701
Ecuador     Apartment   576   554
            House       588  1194
El-salvador Apartment   109   391
            House        91   810
Guatemala   Apartment   268  1002
            House        70   845
Honduras    Apartment    41   290
            House         6   468
Mexico      Apartment  1504  1039
            House      1298  4544
Nicaragua   Apartment    22    96
            House       239  1037
Panama      Apartment  1417  1744
            House       703

In [125]:
print("\nsources")
print(clean_scrap.groupby('country')['source'].unique().to_string())


sources
country
Argentina            [Properati]
Bolivia              [Infocasas]
Brazil             [QuintoAndar]
Chile                     [Yapo]
Colombia             [Properati]
Costa-rica         [Encuentra24]
Dominicana         [Encuentra24]
Ecuador              [Properati]
El-salvador        [Encuentra24]
Guatemala          [Encuentra24]
Honduras           [Encuentra24]
Mexico         [Pincali, iCasas]
Nicaragua          [Encuentra24]
Panama             [Encuentra24]
Paraguay             [Infocasas]
Peru                 [Properati]
Uruguay              [Infocasas]


In [126]:
## Save clean_data
clean_scrap.to_csv(path + "/clean_scrap.csv",  index=False, encoding="utf-8-sig")

### Summary stats

In [ ]:
summary_by_country_type_operation = clean_scrap.groupby(['country', 'type', 'operation']).agg(
    listings=('price', 'size'),
    avg_price_usd=('price_usd', 'mean'),
    med_price_usd=('price_usd', 'median'),
    min_price_usd=('price_usd', 'min'),
    max_price_usd=('price_usd', 'max'),
    avg_size=('size', 'mean'),
    med_size=('size', 'median'),
    min_size=('size', 'min'),
    max_size=('size', 'max'),
    avg_price_per_sq_meter=('price_per_sq_meter', 'mean'),
).sort_values(['country', 'listings'], ascending=[True, False])

summary_by_country_type_operation = summary_by_country_type_operation.round({
    'avg_price_usd': 2,
    'med_price_usd': 2,
    'min_price_usd': 2,
    'max_price_usd': 2,
    'avg_size': 2,
    'med_size': 2,
    'min_size': 2,
    'max_size': 2,
    'avg_price_per_sq_meter': 2,
})

for country, country_summary in summary_by_country_type_operation.groupby(level='country'):
    country_table = country_summary.reset_index(level='country', drop=True).reset_index()
    country_table['group'] = country_table['type'] + ' | ' + country_table['operation']
    country_table = country_table.set_index('group')[
        ['listings', 'avg_price_usd', 'med_price_usd', 'min_price_usd', 'max_price_usd',
         'avg_size', 'med_size', 'min_size', 'max_size', 'avg_price_per_sq_meter']
    ]

    print(f"\nSummary for {country}")
    print(country_table.to_string())

    fig, ax = plt.subplots(figsize=(18, max(4, 0.35 * len(country_table))))
    fig.suptitle(f"Summary statistics for {country} (grouped by type and operation)", fontsize=14, y=0.95)
    ax.axis('off')
    table = ax.table(
        cellText=country_table.reset_index().values,
        colLabels=['group', 'listings', 'avg_price', 'med_price', 'min_price', 'max_price',
                   'avg_size', 'med_size', 'min_size', 'max_size', 'avg_price_m2'],
        cellLoc='center',
        loc='center'
    )
    table.auto_set_font_size(False)
    table.set_fontsize(7)
    table.scale(1, 1.1)

    image_file = os.path.join(
        path,
        f"summary_{country.lower().replace(' ', '_').replace('-', '_')}.png",
    )
    plt.savefig(image_file, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved summary image for {country} to: {image_file}")



Summary for Argentina
                  listings  avg_price_usd  med_price_usd  min_price_usd  max_price_usd  avg_size  med_size  min_size   max_size  avg_price_per_sq_meter
group                                                                                                                                                  
House | sell           461     158,647.99     115,000.00          12.00   2,014,770.10  2,220.45    185.00      1.00 596,534.00                1,811.45
Apartment | sell       451     398,247.59     105,000.00         145.06 111,111,111.00  4,429.50     85.00      1.00 943,180.00                2,687.67
Apartment | rent       421       7,795.68         604.42           1.21   1,800,000.00    252.99     62.00      1.00  10,000.00                   59.49
House | rent           265       2,206.81         886.49           0.90      70,000.00    781.69    120.00      1.00 123,123.00                   14.26
Saved summary image for Argentina to: scraping_output/final\summa

### Boxplots per country

In [146]:
distribution_dir = os.path.join(path, "distribution_price_per_sq_meter")
os.makedirs(distribution_dir, exist_ok=True)

for country, country_group in clean_scrap.groupby('country'):
    country_group = country_group.dropna(subset=['price_per_sq_meter'])
    group_data = []
    labels = []

    for (listing_type, operation), group in country_group.groupby(['type', 'operation']):
        values = group['price_per_sq_meter']
        if len(values) < 3:
            continue
        labels.append(f"{listing_type} | {operation}")
        group_data.append(values)

    if not group_data:
        continue

    fig, ax = plt.subplots(figsize=(max(10, len(group_data) * 0.8), 6))
    ax.boxplot(group_data, vert=False, showfliers=False, patch_artist=True,
               boxprops=dict(facecolor='tab:blue', alpha=0.4), medianprops=dict(color='black'))

    from matplotlib.ticker import ScalarFormatter

    ax.set_yticklabels(labels)
    ax.set_xlabel("Price per square meter (USD)")
    ax.set_title(
        f"Price per square meter by type and operation for {country}",
        fontsize=12,
    )
    ax.grid(axis='x', alpha=0.3)
    ax.set_xscale('log')
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.xaxis.get_major_formatter().set_scientific(False)
    ax.ticklabel_format(style='plain', axis='x')
    ax.text(
        0.99,
        -0.12,
        "Log scale; outliers omitted for clarity.",
        transform=ax.transAxes,
        ha='right',
        va='top',
        fontsize=9,
        color='gray',
    )

    distribution_file = os.path.join(
        distribution_dir,
        f"distribution_price_per_sq_meter_{country.lower().replace(' ', '_').replace('-', '_')}.png",
    )
    fig.tight_layout()
    plt.savefig(distribution_file, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved boxplot distribution for {country} to: {distribution_file}")

Saved boxplot distribution for Argentina to: scraping_output/final\distribution_price_per_sq_meter\distribution_price_per_sq_meter_argentina.png
Saved boxplot distribution for Bolivia to: scraping_output/final\distribution_price_per_sq_meter\distribution_price_per_sq_meter_bolivia.png
Saved boxplot distribution for Brazil to: scraping_output/final\distribution_price_per_sq_meter\distribution_price_per_sq_meter_brazil.png
Saved boxplot distribution for Chile to: scraping_output/final\distribution_price_per_sq_meter\distribution_price_per_sq_meter_chile.png
Saved boxplot distribution for Colombia to: scraping_output/final\distribution_price_per_sq_meter\distribution_price_per_sq_meter_colombia.png
Saved boxplot distribution for Costa-rica to: scraping_output/final\distribution_price_per_sq_meter\distribution_price_per_sq_meter_costa_rica.png
Saved boxplot distribution for Dominicana to: scraping_output/final\distribution_price_per_sq_meter\distribution_price_per_sq_meter_dominicana.png
S